# Ders 01 — Görüntü aslında bir NumPy dizisidir

**Bu dersin amacı:** OpenCV ve NumPy'da öğrendiklerini sağlamlaştırmak. Sonraki bütün dersler (renk, eşikleme, kenar, kontur...) bu temelin üstüne kurulacak.

**Nasıl çalışmalısın:**
1. Her kod hücresini çalıştırmadan önce **ne çıkacağını tahmin et** (kafandan ya da kağıda). Sonra `Shift+Enter` ile çalıştır.
2. Tahminin yanlış çıktıysa işte orası öğrenme anıdır: *neden* farklı çıktığını anlamadan geçme.
3. Sayıları değiştir, bozmaya çalış. Hiçbir şey kırılmaz.
4. En sondaki alıştırmaları **kendin** yap. Takıldığında bana sor, çözdükten sonra da bana göster, birlikte bakalım.

> Veri setimiz gelene kadar kod içinde kendi "sahte İHA görüntümüzü" üreteceğiz. Böylece her pikselin ne olduğunu bildiğimiz için sonuçları kontrol etmek kolay olacak.

In [ ]:
# --- Kütüphaneleri içe aktarma ---
import cv2                        # OpenCV: görüntü okuma, işleme, çizim fonksiyonları
import numpy as np                # NumPy: görüntüler aslında NumPy dizisidir, hesapları bununla yaparız
import matplotlib.pyplot as plt   # Matplotlib: görüntüleri notebook içinde göstermek için

# Kurulu sürümleri yazdır (hata ayıklarken işe yarar)
print("OpenCV:", cv2.__version__)
print("NumPy :", np.__version__)

# --- Yardımcı fonksiyon: görüntüyü doğru renklerde göster ---
def goster(img, baslik="", boyut=(8, 5)):
    """OpenCV görüntüsünü (BGR veya gri) matplotlib ile doğru renklerde gösterir."""
    plt.figure(figsize=boyut)              # yeni bir çizim alanı aç; boyut = (genişlik, yükseklik) inç
    if img.ndim == 2:                      # ndim = boyut sayısı; 2 ise tek kanallı (gri) görüntü
        plt.imshow(img, cmap="gray",       # cmap="gray": sayıları gri tonlarla boya
                   vmin=0, vmax=255)       # 0 = siyah, 255 = beyaz olarak sabitle (otomatik ölçekleme yapma)
    else:                                  # 3 boyutlu ise renkli (yükseklik, genişlik, 3)
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # OpenCV BGR tutar, matplotlib RGB bekler -> çevir
        plt.imshow(rgb)
    plt.title(baslik)                      # üstüne başlık yaz
    plt.axis("off")                        # eksen çizgilerini/sayılarını gizle
    plt.show()                             # çizimi ekrana bas

## 1. Görüntü = sayı tablosu

Gri tonlu bir görüntü, her hücresi bir pikselin **parlaklığı** olan 2 boyutlu bir tablodur:
- `0` = siyah, `255` = beyaz, arası gri tonlar.

Aşağıda 4 satır × 6 sütunluk minik bir "görüntü" elle yazıyoruz. Çalıştırmadan önce tahmin et: nasıl bir şekil görünecek?

In [ ]:
# np.array ile elle 4 satır x 6 sütunluk bir tablo (= gri görüntü) oluşturuyoruz
# Her sayı bir pikselin parlaklığı: 0 = siyah, 255 = beyaz
minik = np.array([
    [  0,   0, 255, 255,   0,   0],   # 1. satır (y = 0)
    [  0, 255, 255, 255, 255,   0],   # 2. satır (y = 1)
    [  0, 255, 255, 255, 255,   0],   # 3. satır (y = 2)
    [  0,   0, 255, 255,   0,   0],   # 4. satır (y = 3)
], dtype=np.uint8)                    # dtype=uint8: her hücre 0-255 arası tam sayı (görüntülerin standart türü)

print(minik)                          # dizinin ham sayılarını yazdır
print("shape:", minik.shape)          # (satır, sütun) = (yükseklik, genişlik) -> (4, 6)
print("dtype:", minik.dtype)          # veri türü -> uint8

# interpolation="nearest": pikselleri büyütürken yumuşatma yapma, kare kare göster
plt.imshow(minik, cmap="gray", interpolation="nearest")
plt.title("4x6 piksellik görüntü")
plt.show()

**Hatırla:**
- `shape` → `(yükseklik, genişlik)`. Önce **satır (y)**, sonra **sütun (x)**.
- `dtype=uint8` → *unsigned 8-bit integer*, yani 0 ile 255 arasındaki tam sayılar. Görüntülerin neredeyse hepsi bu türde gelir.

### ⚠️ Tuzak 1: uint8 taşması

uint8 yalnızca 0–255 tutabildiği için 255'i aşınca sayı **başa sarar**. Tahmin et: `250 + 10` kaç çıkacak?

In [ ]:
# np.full(şekil, değer, dtype): verilen şekilde, her hücresi aynı değer olan dizi üretir
a = np.full((2, 3), 250, dtype=np.uint8)   # 2x3'lük, her pikseli 250 olan minik görüntü
b = np.full((2, 3), 10,  dtype=np.uint8)   # 2x3'lük, her pikseli 10 olan minik görüntü

print("NumPy ile toplama:")
print(a + b)            # NumPy uint8'de 255'i aşınca başa sarar: 260 - 256 = 4
print("cv2.add ile toplama:")
print(cv2.add(a, b))    # OpenCV 255'te durdurur (doygunluk / saturation)

`260 - 256 = 4`. NumPy sayıyı sarar. `cv2.add` ise 255'te keser; buna **saturation (doygunluk)** denir.

**Pratik ders:** Görüntünün parlaklığını artırırken `img + 50` yazarsan parlak pikseller birden kararır. `cv2.add(img, 50)` kullan ya da önce `int16`/`float` türüne çevir.

---
## 2. Renkli görüntü ve BGR

Renkli görüntünün 3 kanalı vardır, `shape` değeri `(yükseklik, genişlik, 3)` olur.
**OpenCV kanalları B-G-R (Mavi, Yeşil, Kırmızı) sırasında tutar.** Matplotlib ve neredeyse bütün diğer kütüphaneler ise RGB kullanır.

Şimdi basit bir kuşbakışı sahne çizelim: tarla, bir yol, binalar ve araçlar. Her satırı oku; hepsi ya **dilimleme (slicing)** ya da çizim fonksiyonu.

In [ ]:
# --- Görüntünün boyutu ---
YUKSEKLIK, GENISLIK = 400, 600     # 400 satır (y), 600 sütun (x)

# --- Renkler (DİKKAT: OpenCV sırası B, G, R) ---
TARLA = (60, 140, 70)      # B=60,  G=140, R=70  -> yeşil
YOL   = (128, 128, 128)    # üç kanal eşit        -> gri
BINA  = (40, 60, 150)      # R en yüksek          -> kırmızımsı çatı
ARAC  = (230, 200, 30)     # B en yüksek          -> mavimsi araç

# np.zeros: her yeri 0 (siyah) olan boş bir görüntü. Şekil = (yükseklik, genişlik, 3 kanal)
sahne = np.zeros((YUKSEKLIK, GENISLIK, 3), dtype=np.uint8)
sahne[:] = TARLA                     # [:] = "tüm pikseller" -> hepsini tarla rengine boya

# Dilimleme ile yol çizme: sahne[satır_aralığı, sütun_aralığı]
sahne[180:230, :] = YOL              # yatay yol: 180-229. satırlar, tüm sütunlar (: = hepsi)
sahne[:, 400:440] = YOL              # dikey yol: tüm satırlar, 400-439. sütunlar

# cv2.rectangle(görüntü, sol_üst_köşe(x, y), sağ_alt_köşe(x, y), renk, kalınlık)
# kalınlık = -1 -> dikdörtgenin içini doldur (pozitif sayı olsaydı sadece çerçeve çizerdi)
cv2.rectangle(sahne, (50, 40),   (150, 130), BINA, -1)   # sol üst bina
cv2.rectangle(sahne, (200, 60),  (320, 150), BINA, -1)   # orta üst bina
cv2.rectangle(sahne, (470, 260), (570, 360), BINA, -1)   # sağ alt bina

# Yatay yolun üstüne 3 araç: her biri 30 piksel geniş, 15 piksel yüksek
for x in (60, 170, 300):                                  # x = aracın sol kenarı
    cv2.rectangle(sahne, (x, 190), (x + 30, 205), ARAC, -1)
# Dikey yolun üstüne 1 araç (dik duruyor: 15 geniş, 30 yüksek)
cv2.rectangle(sahne, (410, 300), (425, 330), ARAC, -1)

print("shape:", sahne.shape, "| dtype:", sahne.dtype)    # (400, 600, 3) | uint8
goster(sahne, "Sahte İHA görüntüsü (doğru renkler)")      # yardımcı fonksiyonumuz BGR->RGB çevirir

Aynı görüntüyü dönüştürmeden doğrudan matplotlib'e verirsek ne olur? Tahmin et: tarla hangi renkte görünecek?

In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(sahne)      # BGR diziyi ÇEVİRMEDEN veriyoruz; matplotlib onu RGB sanacak
plt.title("YANLIŞ: BGR görüntü RGB sanılarak gösterildi")
plt.axis("off")
plt.show()
# Sonuç: mavi ve kırmızı kanallar yer değiştirir, renkler tuhaf görünür

Mavi ve kırmızı kanallar yer değiştirdi. Görüntü işlemede "renkler tuhaf çıktı" şikayetinin 1 numaralı sebebi budur. Bu yüzden `goster()` fonksiyonumuz her seferinde `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` yapıyor.

---
## 3. Piksel erişimi: `[y, x]` ama `(x, y)`

### ⚠️ Tuzak 2: iki farklı koordinat sırası
- **NumPy indeksleme:** `img[y, x]` → önce satır, sonra sütun.
- **OpenCV çizim fonksiyonları:** `(x, y)` → önce yatay, sonra dikey.

Yukarıda binayı `cv2.rectangle(sahne, (50, 40), ...)` ile çizdik: x=50, y=40. O köşedeki pikseli NumPy'da okumak için **`sahne[40, 50]`** yazmalıyız.

In [ ]:
# NumPy indeksleme: sahne[y, x] -> önce SATIR (dikey), sonra SÜTUN (yatay)
print("Bina köşesi  sahne[40, 50]  =", sahne[40, 50])    # y=40, x=50 -> binanın sol üst köşesi
print("Ters yazarsak sahne[50, 40] =", sahne[50, 40])    # y=50, x=40 -> bambaşka bir piksel!
print("Yolun üstü   sahne[200, 10] =", sahne[200, 10])   # y=200 yatay yolun içinde
print("Tek kanal: B değeri          =", sahne[200, 10, 0])  # 3. indeks = kanal: 0=B, 1=G, 2=R

`sahne[50, 40]` ise **tarla** rengi döndü: x=40 noktası binanın solunda, dışarıda kalıyor. Koordinatları ters yazınca tamamen başka bir piksele baktık.
Bu hata çok sinsidir, çünkü hata mesajı çıkmaz. Kare bölgelerde ya da x ile y birbirine yakınken "şans eseri" doğru sonuç bile verebilir.
**Alışkanlık edin:** Kodda `y, x` değişken adlarını açıkça kullan.

## 4. Dilimleme ile kırpma (crop)

İHA görüntüleri çok büyüktür (örneğin 6000×4000 piksel). Sürekli belli bir bölgeyi kesip inceleyeceğiz. Kırpma sadece dilimlemedir: `img[y1:y2, x1:x2]`.

In [ ]:
# Kırpma = dilimleme: img[y1:y2, x1:x2]
# Not: bitiş değeri DAHİL DEĞİL -> 170:240 demek 170, 171, ..., 239. satırlar
y1, y2 = 170, 240        # dikeyde: yolun biraz üstünden biraz altına
x1, x2 = 40, 350         # yatayda: ilk üç aracı içine alacak kadar
yol_bolgesi = sahne[y1:y2, x1:x2]

print("Kırpılan bölge shape:", yol_bolgesi.shape)   # (240-170, 350-40, 3) = (70, 310, 3)
goster(yol_bolgesi, "Kırpılmış bölge: yol ve araçlar", boyut=(8, 2.5))

### ⚠️ Tuzak 3: Dilim bir kopya değil, bir **görünümdür (view)**

`yol_bolgesi` yeni bir görüntü değil, `sahne`nin içine açılmış bir pencere. Birini değiştirirsen diğeri de değişir. Tahmin et: aşağıdaki kod `sahne`yi etkileyecek mi?

In [ ]:
deneme = sahne.copy()                 # .copy(): bellekte YENİ bir dizi -> asıl sahne güvende kalır
pencere = deneme[0:100, 0:100]        # dilim = view: yeni dizi değil, "deneme"nin sol üst köşesine açılan pencere
pencere[:] = (0, 0, 255)              # penceredeki tüm pikselleri kırmızı yap (BGR'de kırmızı = (0, 0, 255))

# Sadece "pencere"yi boyadık ama "deneme"yi gösteriyoruz: sol üst köşe kırmızı olmuş olacak
goster(deneme, "Pencereyi boyadık ve orijinal de değişti!")

Bağımsız bir parça istiyorsan `.copy()` kullan: `parca = img[y1:y2, x1:x2].copy()`.
Bu davranış bazen işine yarar (bir bölgeyi yerinde düzenlemek), bazen de saatlerce bug aratır.

---
## 5. Kanallar ve gri tonlama

In [ ]:
# cv2.split: 3 kanallı görüntüyü 3 ayrı 2B diziye ayırır (her biri 400x600)
b, g, r = cv2.split(sahne)

# cv2.cvtColor ile renk dönüşümü: BGR -> tek kanallı gri
gri = cv2.cvtColor(sahne, cv2.COLOR_BGR2GRAY)

# 1 satır, 4 sütunluk yan yana grafik alanı aç; "eksen" 4 alt grafiğin listesi
fig, eksen = plt.subplots(1, 4, figsize=(16, 3))

# zip: üç listeyi eşleştirir -> (ilk eksen, b, "B (mavi)"), (ikinci eksen, g, ...), ...
for ax, kanal, ad in zip(eksen, (b, g, r, gri), ("B (mavi)", "G (yeşil)", "R (kırmızı)", "Gri")):
    ax.imshow(kanal, cmap="gray", vmin=0, vmax=255)   # her kanalı gri tonlarla göster
    ax.set_title(ad)                                  # alt grafiğin başlığı
    ax.axis("off")                                    # eksenleri gizle
plt.show()

print("gri shape:", gri.shape)   # (400, 600) -> kanal boyutu yok, çünkü tek kanal

Her kanal kendi başına gri bir görüntüdür. Parlak yer "o renkten çok var" demektir. Araçlar **B** kanalında neden bembeyaz? `ARAC = (230, 200, 30)` değerine bak.

Bu fikir ileride çok işimize yarayacak: bitki örtüsü yeşil kanalda, sular mavi kanalda öne çıkar.

---
## 6. Maske: "hangi pikseller şu koşulu sağlıyor?"

NumPy'da bir karşılaştırma yaptığında sonuç True/False'lardan oluşan bir dizi olur. Buna **maske** denir.
Soru: *Görüntünün yüzde kaçı yol?*

In [ ]:
# sahne == YOL -> her pikselin her kanalını YOL rengiyle karşılaştırır
#   sonuç şekli (400, 600, 3), içi True/False
# np.all(..., axis=2) -> 2 numaralı eksen (kanallar) boyunca "üçü de True mu?"
#   sonuç şekli (400, 600): piksel yol rengindeyse True
yol_maskesi = np.all(sahne == YOL, axis=2)

print("maske shape:", yol_maskesi.shape, "| dtype:", yol_maskesi.dtype)   # (400, 600) | bool
print("yol piksel sayısı:", yol_maskesi.sum())       # True = 1 sayılır -> toplam = yol piksel adedi
print(f"alanın %{100 * yol_maskesi.mean():.1f}'i yol")  # mean = True oranı; :.1f -> virgülden sonra 1 basamak

# Maskeyi görüntü olarak göstermek için: bool -> uint8 (0/1) -> x255 (0/255 = siyah/beyaz)
goster(yol_maskesi.astype(np.uint8) * 255, "Yol maskesi (beyaz = yol)")

Birkaç şeye dikkat et:
- `sahne == YOL` → `(400, 600, 3)` boyutlu True/False dizisi (her kanal ayrı karşılaştırılır; bu **broadcasting**).
- `np.all(..., axis=2)` → 3 kanalın hepsi True mu? Sonuç `(400, 600)`.
- `True` sayılırken 1 kabul edilir, bu yüzden `.sum()` piksel sayısını, `.mean()` oranı verir.
- Araçların altındaki yol maskede **delik** olarak görünüyor. Çünkü o pikseller artık araç rengi.

> Gerçek İHA fotoğraflarında hiçbir piksel *tam olarak* aynı renkte olmaz (ışık, gölge, gürültü). Bu yüzden ileride "tam eşitlik" yerine **renk aralıkları** (`cv2.inRange`) ve **HSV renk uzayı** kullanacağız. Ders 02'nin konusu bu.

## 7. Kaydetme

In [ ]:
import os
# Çıktı klasörünü oluştur; exist_ok=True -> klasör zaten varsa hata verme
os.makedirs("../outputs", exist_ok=True)

# cv2.imwrite(dosya_yolu, görüntü): diske kaydeder. BGR bekler, bu yüzden çevirmeye gerek yok
# Dosya uzantısı (.png, .jpg) formatı belirler
cv2.imwrite("../outputs/ders01_sahne.png", sahne)
cv2.imwrite("../outputs/ders01_yol_maskesi.png", yol_maskesi.astype(np.uint8) * 255)  # maskeyi 0/255'e çevirip kaydet

# Geri okuyup aynı mı diye kontrol et
geri = cv2.imread("../outputs/ders01_sahne.png")              # BGR olarak okur
print("Geri okundu:", geri.shape,
      "| aynı mı?", np.array_equal(geri, sahne))              # iki dizi piksel piksel aynı mı? PNG kayıpsız -> True

Not: `cv2.imread` dosyayı bulamazsa hata **vermez**, sessizce `None` döndürür. Gerçek veriyle çalışırken her zaman şunu kontrol edeceğiz:
```python
img = cv2.imread(yol)
assert img is not None, f"Okunamadı: {yol}"
```

---
## 🧠 Kendini test et
Önce cevabı kafanda ver, sonra aç.

<details><summary>1. 6000×4000 bir fotoğrafı okudun. <code>img.shape</code> ne olur?</summary>

`(4000, 6000, 3)`. Önce yükseklik gelir.
</details>

<details><summary>2. <code>img[10, 20]</code> hangi pikseli verir?</summary>

y=10 (satır), x=20 (sütun) noktasındaki pikseli verir. Değer BGR sırasında bir üçlüdür.
</details>

<details><summary>3. <code>np.uint8(200) + np.uint8(100)</code> kaç eder?</summary>

`44` eder (300 − 256). `cv2.add` ile 255 çıkar.
</details>

<details><summary>4. <code>parca = img[0:50, 0:50]</code>; <code>parca[:] = 0</code>. <code>img</code> değişir mi?</summary>

Değişir, çünkü dilim bir view. Bağımsız parça için `.copy()` gerekir.
</details>

---
## ✍️ Alıştırmalar (kendin yap)

Her alıştırmanın altındaki boş hücreye yaz. Takılırsan önce 5 dakika dene, sonra bana sor.

**A1.** `sahne` görüntüsünde sağ alttaki binayı kırp (koordinatları çizim kodundan bul) ve göster. `shape` değeri ne?

In [ ]:
# A1

**A2.** `sahne`nin kopyasını al. Yol ile tarlanın sınırına, yani y=178 ve y=231'e boydan boya 2 piksel kalınlığında sarı çizgiler çiz. Önce dilimlemeyle yap, sonra `cv2.line` ile. (İpucu: BGR'de sarı nasıl yazılır?)

In [ ]:
# A2

**A3.** Yol maskesine benzer bir **araç maskesi** oluştur ve sahnede kaç araç pikseli olduğunu, alanın yüzde kaçını kapladıklarını yazdır.

In [ ]:
# A3

**A4.** Görüntünün parlaklığını 80 artır. Bir kez `sahne + 80`, bir kez `cv2.add(sahne, (80, 80, 80, 0))` ile yap ve ikisini yan yana göster. Hangi bölgeler bozuldu ve **neden**?

In [ ]:
# A4

**A5 (zor).** Sahnede kaç ayrı araç olduğunu bulmayı dene. Maskeyi kullanabilirsin ama henüz öğrenmediğimiz fonksiyonları kullanmak yok. Nasıl bir mantık kurardın? Kod yazamasan bile fikrini yaz. Bunu ileride `cv2.connectedComponents` ve konturlarla çözeceğiz.

In [ ]:
# A5